# RSNA Knee: Verified 0.956 SWA / R7 Ensemble

This notebook combines **50% SWA CoAtNet + 50% R7 ConvNeXt-small**. Competition submission **57143234**, generated by the original **v1**, achieved a **public leaderboard score of 0.956**.
This update improves the documentation and code organization while retaining the scored model weights, preprocessing, R7 postprocessing, and rank fusion.

## Workflow

1. Review the model branches and required datasets.
2. Check the configuration and verify the inference source files.
3. Run SWA and R7 inference in separate processes.
4. Compute percentile ranks for each finding and blend them with equal weights.
5. Validate study IDs, column names, and prediction values; export `submission.csv`.

The notebook presents the main workflow in short code cells. Complete model implementations are available as individual `.py` files in the attached code dataset.
Inference does not read training labels or test reports. Logs, asset manifests, and validation receipts are saved for reproducibility.


## 1. Models and Confirmed Results

| Branch | Architecture and input | Final weight |
|---|---|---:|
| SWA Raptor | CoAtNet with multi-slice MIL and the original anatomical mirror TTA | 50% |
| R7 | Four ConvNeXt-small folds; 352-pixel crops, 3.5 mm physical slice spacing, and v4 anatomical views | 50% |

R7 retains its original Effusion volume stacker and Baker's state stacker. The four folds are combined by the author's inference implementation before per-finding rank fusion with SWA.

| Official submission from this account | Public LB |
|---|---:|
| SWA 50% + S6 50% | 0.954 |
| **This pipeline: SWA 50% + R7 50%** | **0.956** |

The **0.956** score belongs to the official submission from **v1**. The readability refactor in **v2** reproduced its visible predictions and final CSV byte for byte. This English documentation update keeps every executable code cell identical to v2.


## 2. Configuration and Required Datasets

Use a T4 GPU with Internet disabled. The main configuration is collected in the cell below.

| Dataset | Purpose |
|---|---|
| `rsna-knee-abnormality-detection` | Test DICOM studies, series metadata, and submission schema |
| `nartaa/rsna-knee-publication-swa-weights-20261007` | The scored SWA checkpoint |
| `heliosli/rsna-knee-compact-r7cnxfsss` v1 | Four R7 checkpoints and their postprocessing assets |
| `heliosli/rsna-orthofoundation-fold0-epoch7-20261004` | DICOM decoding, rendering, anatomical localization, and offline dependencies |
| `picond/rsna-knee-r7-readable-inference-code` v1 | Readable inference source files and workflow utilities |


In [ ]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

NB_T0 = time.time()
WORK = Path('/kaggle/working')
WORK.mkdir(exist_ok=True)
DEADLINE = NB_T0 + 8.3 * 3600
BLEND_WEIGHTS = {'swa': 0.5, 'r7': 0.5}
assert sum(BLEND_WEIGHTS.values()) == 1

def find_input(slug, required):
    root = Path('/kaggle/input')
    candidates = [root / slug, root / 'competitions' / slug]
    candidates += list(root.glob('datasets/*/' + slug))
    candidates += list(root.glob('datasets/*/' + slug + '/*'))
    found = list(dict.fromkeys(p for p in candidates if (p / required).is_file()))
    assert len(found) == 1, (slug, found)
    return found[0]

DATA = find_input('rsna-knee-abnormality-detection', 'sample_submission.csv')
CODE_ROOT = find_input('rsna-knee-r7-readable-inference-code', 'code_manifest.json')
sample = pd.read_csv(DATA / 'sample_submission.csv', dtype={'StudyInstanceUID': str})
IDS = sample.StudyInstanceUID.tolist()
LABELS = sample.columns.tolist()[1:]
assert len(IDS) == len(set(IDS)) and len(LABELS) == 12
display(pd.DataFrame([{'model': name, 'weight': weight} for name, weight in BLEND_WEIGHTS.items()]))
print(f'Test studies: {len(IDS)}; findings: {len(LABELS)}')


## 3. Verify Source Code and Model Versions

First verify the fixed SHA256 of the code manifest, then verify every listed source file.
The model preparation stages also verify the original hashes of the SWA checkpoint, all four R7 folds, anatomical localization weights, and postprocessing assets.

| Source file | Responsibility |
|---|---|
| `raptor_run.py` | Original v1 SWA model, DICOM reader, and inference pipeline |
| `prepare_r7.py` | R7 dependencies and asset preparation |
| `compact_infer.py` / `compact_train.py` | Original R7 inference and model definitions |
| `probe_dual.py` | Dual-GPU inference controller |
| `runtime.py` | Process management, study-ID alignment, blending, and CSV export |

The full implementations are readable text files rather than large string dictionaries embedded in the notebook.


In [ ]:
import hashlib

EXPECTED_CODE_MANIFEST_SHA256 = 'd96fa467ce2d0f7127b5def38c7fc9de7763504472576b29f40de2c4467cd910'
manifest_file = CODE_ROOT / 'code_manifest.json'
assert hashlib.sha256(manifest_file.read_bytes()).hexdigest() == EXPECTED_CODE_MANIFEST_SHA256
code_manifest = json.loads(manifest_file.read_text())
for name, digest in code_manifest['sha256'].items():
    assert hashlib.sha256((CODE_ROOT / name).read_bytes()).hexdigest() == digest, name

sys.path.insert(0, str(CODE_ROOT))
from runtime import (
    blend_predictions, prepare_r7_assets, read_predictions,
    run_process, run_r7, save_predictions,
)
expected_r7 = json.loads((CODE_ROOT / 'expected_r7_manifest.json').read_text())
assert expected_r7['bundle_id'] == '8c7b666e27dc0e8f'
(WORK / 'code_manifest.json').write_text(json.dumps(code_manifest, indent=2))
print(f'Code verified: {len(code_manifest["sha256"])} files; test studies: {len(IDS)}')


## 4. Run SWA CoAtNet Inference

Run the original v1 SWA implementation with its anatomical mirror TTA.
The isolated process releases GPU memory when it finishes. Its complete log is saved as `swa_inference.log`.

`raptor_raw.csv` contains the raw probabilities used for ensemble fusion. `raptor_sub.csv` contains the SWA branch's own ranked predictions.


In [ ]:
TIMINGS = {}
TIMINGS['swa'] = run_process(
    [sys.executable, '-u', str(CODE_ROOT / 'raptor_run.py')],
    WORK / 'swa_inference.log', min(DEADLINE, time.time() + 2 * 3600),
)
swa = read_predictions(WORK / 'raptor_raw.csv', IDS, LABELS)
assert not (swa.to_numpy() == 0.5).all(axis=1).any(), 'SWA has failed-study placeholders'
print(f'SWA complete: {len(swa)} studies, {len(swa.columns)} findings')


## 5. Run the Four R7 Folds

Prepare the same checkpoints, anatomical localization, laterality checks, dependencies, and source files as v1.
Keep R7's original Effusion and Baker's postprocessing. Additional R7 mirror TTA remains disabled, matching the scored configuration.

All four folds must load successfully. The stage writes `r7_stacked.csv` and a detailed inference receipt.
A failed stage stops execution and preserves its log for diagnosis.


In [ ]:
r7_context = prepare_r7_assets(CODE_ROOT, NB_T0, expected_r7)
prediction_dir, r7_receipt, TIMINGS['r7'] = run_r7(r7_context, WORK, DEADLINE)
r7 = read_predictions(WORK / 'r7_stacked.csv', IDS, LABELS)
assert r7_receipt['scored'] == len(IDS)
assert r7_receipt['fallback_studies'] == 0
r7.to_csv(WORK / 'ours_sub.csv', float_format='%.9g')
display(pd.DataFrame([{
    'worker': index,
    'fold_models': len(worker['models']),
    'scored_studies': worker['scored'],
    'status': worker['status'],
} for index, worker in enumerate(r7_receipt['workers'])]))
print('R7 complete: four folds, zero fallback; original postprocessing retained')


## 6. Blend Per-Finding Ranks with Equal Weights

The two models can produce probabilities on different scales. Convert each model's predictions to percentile ranks separately for each finding, then combine them:

`final = 0.5 × percentile_rank(SWA) + 0.5 × percentile_rank(R7)`

Exact ties receive average ranks. The blend is not ranked again, and final values are exported as float32, matching the original implementation.
The table below shows how similarly the two branches order studies for each finding.
With only a few visible test studies, these correlations do not estimate hidden-test or validation performance.


In [ ]:
submission = blend_predictions({'swa': swa, 'r7': r7}, BLEND_WEIGHTS)
submission_sha256 = save_predictions(submission, WORK / 'submission.csv')
rank_correlation = swa.rank().corrwith(r7.rank())
display(pd.DataFrame({
    'finding': LABELS,
    'swa_r7_rank_correlation': rank_correlation.to_numpy(),
    'submission_min': submission.min().to_numpy(),
    'submission_max': submission.max().to_numpy(),
}))


## 7. Validate the Submission and Review the Run

Check the order of all test study IDs, the official 12 finding columns, missing values, finite values, and the `[0, 1]` range.
Save the run receipt, elapsed time, and file hash for comparison with the original 0.956 version.

| Output file | Purpose |
|---|---|
| `submission.csv` | Final 50/50 ensemble submission |
| `raptor_raw.csv` / `ours_sub.csv` | Raw predictions from the two model branches |
| `r7_inference_receipt.json` | Fold loading and actual worker execution details |
| `readability_receipt.json` | Recipe, source version, timings, and final SHA256 |
| `swa_inference.log` / `r7_inference.log` | Complete inference logs |


In [ ]:
final = pd.read_csv(WORK / 'submission.csv', dtype={'StudyInstanceUID': str})
assert final.StudyInstanceUID.tolist() == IDS
assert final.columns.tolist() == sample.columns.tolist()
assert final.StudyInstanceUID.is_unique
assert np.isfinite(final[LABELS].to_numpy()).all()
assert ((final[LABELS] >= 0) & (final[LABELS] <= 1)).all().all()
receipt = {
    'status': 'COMPLETE', 'recipe': 'swa50_r750', 'weights': BLEND_WEIGHTS,
    'rows': len(IDS), 'source_version': 1,
    'scored_submission_ref': 57143234, 'confirmed_source_public_score': 0.956,
    'r7_bundle_id': expected_r7['bundle_id'],
    'r7_postprocessing_retained': True, 'r7_fallback_studies': 0,
    'submission_sha256': submission_sha256, 'timings': TIMINGS,
    'total_seconds': round(time.time() - NB_T0, 2),
}
(WORK / 'readability_receipt.json').write_text(json.dumps(receipt, indent=2))
display(final.head())
display(pd.DataFrame([{'stage': stage, 'seconds': seconds} for stage, seconds in TIMINGS.items()]))
print('Submission validated:', final.shape, submission_sha256)


## Version History and Reproducibility

**v1:** Official competition submission **57143234** achieved **Public LB 0.956**.

**v2:** The readability refactor completed successfully. R7 raw probabilities and the final submission CSV were identical to v1 on the visible test studies. All four folds ran with zero fallback.

**v3:** All notebook documentation is in English. Every executable code cell is unchanged from the verified v2.

The visible-output comparison does not replace hidden-test scoring. The historical 0.956 score is attributed to its original official submission.

Original public pipeline: [Helios R7](https://www.kaggle.com/code/heliosli/0-957-latest-dataset-score-in-one-click).
Changes to blend weights, checkpoints, crops, TTA, or postprocessing should be tracked as separate experiments.


## Credits and required acknowledgement

Built on dreaddevelopment's [Raptor CoAtNet/MIL inference recipe and public baseline weights](https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense). These are our retrained checkpoints, rather than those baseline weights. Weak-label sources include [stevenleehans](https://www.kaggle.com/datasets/stevenleehans/rsna-knee-llm-report-labels), [pilkwang](https://www.kaggle.com/datasets/pilkwang/rsna-knee-llm-labels), and [riadmohamed42's JEV labels and verification](https://www.kaggle.com/code/riadmohamed42/jev-knee-labels-verification), combined with our report-derived labels. Source reports and study-level label tables are not included here. The CoAtNet backbone comes from timm. Competition data is provided under the [RSNA Knee competition rules](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/rules).

Data and/or research tools used in the preparation of this manuscript were obtained and analyzed from the controlled access datasets distributed from the Osteoarthritis Initiative (OAI), a data repository housed within the NIMH Data Archive (NDA). OAI is a collaborative informatics system created by the National Institute of Mental Health and the National Institute of Arthritis, Musculoskeletal and Skin Diseases (NIAMS) to provide a worldwide resource to quicken the pace of biomarker identification, scientific investigation and OA drug development. Dataset identifier(s): 10.15154/0hcg-f676.

[Shared NDA Study](https://nda.nih.gov/study.html?id=3407) | [DOI: 10.15154/0hcg-f676](https://doi.org/10.15154/0hcg-f676). Official acknowledgement source: [NDA manuscript preparation](https://nda.nih.gov/nda/manuscript-preparation).
